# 🔬 Band Gap Predictor — Ternary Compounds Model

**Project:** AI-Driven Calculated Bandgap Data Curation in Materials Project  
**Author:** Akash Rajeshbhai Vaghela — Ruhr-Universität Bochum

---

## How to Use

1. **Run Step 1** once — installs packages and downloads data from GitHub
2. **Run Step 2** once — loads data, engineers features, trains the ExtraTrees model (~1-2 min)
3. **Run Step 3** as many times as you like — enter any formula + DFT band gap → get predicted experimental band gap

---

## Step 1 — Setup

**Run once per session.**

In [ ]:
# Install packages
!pip install xgboost catboost --quiet

# Download data from GitHub (remove old copy if exists)
import shutil, os
if os.path.exists("AI---Driven-calculated-bandgap-datacuration-in-materials-Project-based-onknown-experimental-data"):
    shutil.rmtree("AI---Driven-calculated-bandgap-datacuration-in-materials-Project-based-onknown-experimental-data")
!git clone https://github.com/Akash251041/AI---Driven-calculated-bandgap-datacuration-in-materials-Project-based-onknown-experimental-data.git --quiet

import os
DATA_DIR = 'AI---Driven-calculated-bandgap-datacuration-in-materials-Project-based-onknown-experimental-data'
print("✅ Setup complete! Files available:")
print([f for f in os.listdir(DATA_DIR) if f.endswith('.csv')])

## Step 2 — Load Data, Engineer Features & Train Model

**Run once per session.** Uses your exact pipeline to train the best ExtraTrees model on the Ternary Compounds dataset.

In [ ]:
import os
import pandas as pd
df_cal = pd.read_csv(os.path.join(DATA_DIR, 'ternary_mp.csv'), index_col=False)
df_exp = pd.read_csv(os.path.join(DATA_DIR, 'ternary_exp.csv'), index_col=False)

In [ ]:
import pandas as pd
df = pd.read_csv(os.path.join(DATA_DIR, 'Elements_DB.csv'), index_col=False)

In [ ]:
df.rename(columns={'symbol': 'formula'}, inplace=True)

# Set 'formula' as the index
if 'formula' in df.columns:
    df = df.set_index('formula')

# Clean whitespace
df.index = df.index.astype(str).str.strip()

# Remove duplicates
df = df[~df.index.duplicated(keep='first')]

print(f"Duplicates removed. Unique elements remaining: {len(df)}")

In [ ]:
# Drop ghost keys if they already exist (prevents error on re-run)
for _df in [df_cal, df_exp]:
    for col in ['_key_f', '_key_c']:
        if col in _df.columns:
            _df.drop(columns=[col], inplace=True)

# Create ghost keys
df_cal['_key_f'] = df_cal['formula'].astype(str).str.strip().str.upper()
df_cal['_key_c'] = df_cal['Crystal System'].astype(str).str.strip().str.upper()

df_exp['_key_f'] = df_exp['formula'].astype(str).str.strip().str.upper()
df_exp['_key_c'] = df_exp['Crystal System'].astype(str).str.strip().str.upper()

# Get list of columns to keep from df_exp (everything EXCEPT the join columns)
cols_to_use = [c for c in df_exp.columns if c not in ['formula', 'Crystal System']]
df_exp_subset = df_exp[cols_to_use]

# Merge
print("Merging...")
df_ternary = pd.merge(
    df_cal,
    df_exp_subset,
    on=['_key_f', '_key_c'],
    how='inner'
)

# Cleanup
df_ternary = df_ternary.drop(columns=['_key_f', '_key_c'])

print(f"Matched {len(df_ternary)} rows.")

In [ ]:
import re
formula =  "Zr$_{7}$Fe$_{92}$B"
clean_formula = formula.replace("$","").replace("_","").replace("{","").replace("}","")
element_concenration_pattern = re.compile("([A-Z][a-z]?)([0-9.]*)")
elem_conc_pairs = element_concenration_pattern.findall(clean_formula)
elem_conc_pairs

In [ ]:
def remove_characters(formula, characters_to_remove="$_{} "):
    clean_formula = formula
    for ch in characters_to_remove:
        clean_formula = clean_formula.replace(ch,"")
    return clean_formula

In [ ]:
def convert_to_composition_dict(clean_formula):
    try:
        elem_conc_pairs = element_concenration_pattern.findall(clean_formula)
        comp_dict = {}
        for elem, conc_str in elem_conc_pairs:
            if conc_str=="":
                comp_dict[elem] = 1
            else:
                comp_dict[elem]=float(conc_str)
        conc_sum = 0
        for elem, conc in comp_dict.items():
            conc_sum +=conc
        for elem, conc in comp_dict.items():
            comp_dict[elem] =conc/conc_sum
        return comp_dict
    except ValueError:
        print("Error when converting: ", clean_formula)
        return None

In [ ]:
df_ternary["clean_formula"] = df_ternary["formula"].map(remove_characters)
df_ternary["comp_dict"]     = df_ternary["clean_formula"].map(convert_to_composition_dict)
print(f"Compounds after parsing: {len(df_ternary)}")

In [ ]:
import pandas as pd
import numpy as np
import ast

df_ternary = df_ternary
df = df
parsed_col_name = "comp_dict"
formula_col_name = "formula"

def convert_and_clean_dict(dict_string):
    try:
        comp_dict = ast.literal_eval(str(dict_string))
        if not isinstance(comp_dict, dict):
            return np.nan
        cleaned_dict = {}
        for element, fraction in comp_dict.items():
            if isinstance(element, str):
                cleaned_key = element.strip()
                cleaned_dict[cleaned_key] = fraction
            else:
                return np.nan
        return cleaned_dict
    except (ValueError, SyntaxError, TypeError):
        return np.nan

print("Cleaning df_ternary 'comp_dict' column...")
df_ternary[parsed_col_name] = df_ternary[parsed_col_name].apply(convert_and_clean_dict)
df_ternary = df_ternary.dropna(subset=[parsed_col_name])
print("df_ternary cleaning complete.")

try:
    if df.index.name == formula_col_name:
        df.index = df.index.astype(str).str.strip()
        print(f"df index '{formula_col_name}' was already set. Index cleaned.")
    elif formula_col_name in df.columns:
        df = df.set_index(formula_col_name)
        df.index = df.index.astype(str).str.strip()
        print(f"df index set to '{formula_col_name}' and cleaned.")
    else:
        print(f"Error: Column or Index '{formula_col_name}' not found in df.")
except Exception as e:
    print(f"An error occurred setting up df: {e}")

In [ ]:
def get_weighted_avg(comp_dict, property_name, df_elements):
    total_avg = 0
    try:
        for element, fraction in comp_dict.items():
            prop_value = df_elements.loc[element, property_name]
            if pd.isna(prop_value):
                return np.nan
            total_avg += fraction * prop_value
        return total_avg
    except KeyError:
        return np.nan
    except TypeError:
        return np.nan

def get_difference(comp_dict, property_name, df_elements):
    values = []
    try:
        for element in comp_dict.keys():
            prop_value = df_elements.loc[element, property_name]
            if pd.isna(prop_value):
                return np.nan
            values.append(prop_value)
        if not values:
            return 0
        return max(values) - min(values)
    except (KeyError, AttributeError, TypeError):
        return np.nan

def get_weighted_variance(comp_dict, property_name, mean_val, df_elements):
    if pd.isna(mean_val):
        return np.nan
    total_var = 0
    try:
        for element, fraction in comp_dict.items():
            prop_value = df_elements.loc[element, property_name]
            if pd.isna(prop_value):
                return np.nan
            total_var += fraction * ((prop_value - mean_val) ** 2)
        return total_var
    except (KeyError, AttributeError, TypeError):
        return np.nan

property_list = df.columns

print(f"Starting feature engineering for {len(property_list)} properties...")

df_ternary = df_ternary.copy()

for prop in property_list:
    avg_col_name = f'avg_{prop}'
    df_ternary[avg_col_name] = df_ternary[parsed_col_name].apply(
        lambda comp: get_weighted_avg(comp, prop, df)
    )
    diff_col_name = f'diff_{prop}'
    df_ternary[diff_col_name] = df_ternary[parsed_col_name].apply(
        lambda comp: get_difference(comp, prop, df)
    )
    var_col_name = f'var_{prop}'
    df_ternary[var_col_name] = df_ternary.apply(
        lambda row: get_weighted_variance(row[parsed_col_name], prop, row[avg_col_name], df),
        axis=1
    )

print("--- Feature engineering complete! ---")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor

target_col = 'band_gap_experimenatal (eV)'

df_ternary_cleaned = df_ternary.dropna(subset=[target_col])

feature_cols = [col for col in df_ternary_cleaned.columns if col.startswith(('avg_', 'diff_', 'var_'))]
feature_cols.append('band_gap_calculated (eV)')

X = df_ternary_cleaned[feature_cols]
y = df_ternary_cleaned[target_col]

print(f"Using {len(feature_cols)} features.")
print(f"Training on {X.shape[0]} samples.")

X = X.replace([np.inf, -np.inf], np.nan)
cols_all_nan = X.columns[X.isna().all()]
X = X.drop(columns=cols_all_nan)

imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns)
X = X.clip(np.finfo(np.float32).min, np.finfo(np.float32).max)
X = X.fillna(0)

# Save feature columns for prediction step
FEATURE_COLS = X.columns.tolist()

# Train best model on full dataset
model = ExtraTreesRegressor(n_estimators=300, max_depth=None, random_state=42, n_jobs=-1)
model.fit(X, y)

print(f"\n✅ Model trained and ready!")
print(f"   R² on training data: {model.score(X, y):.4f}")

## Step 3 — Predict Experimental Band Gap

**Edit the two lines below and run this cell.**  
You can run it as many times as you like with different formulas — no retraining needed.

- `INPUT_FORMULA` — any chemical formula (e.g. `GaAs`, `BaTiO3`, `ZnSe`)
- `CALC_BAND_GAP` — the DFT-calculated band gap in eV from Materials Project

In [ ]:
import warnings

# ════════════════════════════════════════════════
#   ✏️  EDIT THESE TWO LINES ONLY
INPUT_FORMULA  = "GaAs"   # ← enter your chemical formula here
CALC_BAND_GAP  = 0.57     # ← enter the DFT-calculated band gap (eV) here
# ════════════════════════════════════════════════

# Parse formula using your exact functions
clean_f = remove_characters(INPUT_FORMULA)
comp    = convert_to_composition_dict(clean_f)

if comp is None:
    print(f"❌ Could not parse formula '{INPUT_FORMULA}'. Please check the format (e.g. GaAs, BaTiO3).")
else:
    # Check for unknown elements
    unknown = [e for e in comp if e not in df.index]
    if unknown:
        warnings.warn(f"⚠️  Warning: element(s) {unknown} not found in Elements_DB.csv. "
                      f"Their properties will be treated as NaN → imputed with column mean. "
                      f"Prediction is a best estimate only.")

    # Build feature vector using your exact feature engineering functions
    feat_row = {}
    for prop in property_list:
        avg_val  = get_weighted_avg(comp, prop, df)
        diff_val = get_difference(comp, prop, df)
        var_val  = get_weighted_variance(comp, prop, avg_val, df)
        feat_row[f'avg_{prop}']  = avg_val
        feat_row[f'diff_{prop}'] = diff_val
        feat_row[f'var_{prop}']  = var_val
    feat_row['band_gap_calculated (eV)'] = CALC_BAND_GAP

    # Align to exact training feature columns
    X_pred = pd.DataFrame([feat_row])
    X_pred = X_pred.reindex(columns=FEATURE_COLS, fill_value=np.nan)
    X_pred = X_pred.replace([np.inf, -np.inf], np.nan)
    X_pred = X_pred.fillna(0)
    X_pred = X_pred.clip(np.finfo(np.float32).min, np.finfo(np.float32).max)

    # Predict
    predicted_exp_bg = model.predict(X_pred)[0]
    predicted_exp_bg = max(predicted_exp_bg, 0.0)  # band gap cannot be negative

    # Display result
    print("=" * 52)
    print(f"  Formula                  : {INPUT_FORMULA}")
    print(f"  Composition              : {comp}")
    print(f"  DFT-Calculated Band Gap  : {CALC_BAND_GAP} eV")
    print(f"  Predicted Experimental   : {predicted_exp_bg:.4f} eV")
    print("=" * 52)
    if unknown:
        print(f"  ⚠️  Note: {unknown} not in Elements_DB — best estimate only.")